# 04 · Neural members: TabM, RealMLP (pytabkit) and the yekenot RealMLP port

**Role in the stack:** diversity that mostly did not pay. `tabm_bag5` CV 0.94417, weight 0.006; `yeke_s42` 0.94595, weight 0.035;
`realmlp_e48x3` 0.9440, weight -0.01; `ftt_s42` (FT-Transformer) 0.9436, weight -0.065 (a contrast column).

## Why they are here
On 09-03 a 0.25 rank-blend of RealMLP into the additive tree was our first real stack gain (+0.0002). Later we learned that
gain was the same interaction/target-encoding signal the depth-5 LightGBM supplies better, and the NN weights went to ~0. We
kept the streams because they are cheap to carry and the stacker decides.

## Inputs
NNs cannot resolve 13,214 exact income values from a float. The wrapper `nnmodel.TEMLP` adds one column: a **cross-fitted
target encoding of exact income** (`sklearn.preprocessing.TargetEncoder(cv=5, smooth=20)`) fitted inside each training fold, plus
the `digits()` columns from `feats.py`. That one column moved RealMLP from 0.9383 to 0.9440.

In [ ]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, '../src')
from cv import load, run_cv, paired, SEED, NFOLDS, OOF, TEST
X, y, Xte, te_id = load()          # data/train.csv, data/test.csv  (kaggle competitions download -c playground-series-s6e9)
print(X.shape, Xte.shape, 'positives %.4f' % y.mean())

In [ ]:
from feats import digits
from nnmodel import TEMLP
from pytabkit import TabM_D_Classifier, RealMLP_TD_Classifier
Xd = digits(X); Xted = digits(Xte).reindex(columns=Xd.columns, fill_value=0)

class NN(TEMLP):
    """TEMLP builds the in-fold income TE; here we only choose the pytabkit model."""
    def __init__(self, kind, n_epochs, seed): super().__init__(n_epochs, seed); self.kind = kind
    def fit(self, X, y):
        k = dict(device='cuda', random_state=self.seed, batch_size=1024, verbosity=0)
        self.m = (TabM_D_Classifier(n_epochs=self.n_epochs, patience=16, **k) if self.kind == 'tabm'
                  else RealMLP_TD_Classifier(n_epochs=self.n_epochs, **k))
        self.m.fit(self._f(X, y), y); return self

EPOCHS = {'tabm': 60, 'realmlp': 48}      # chosen on fold 0; RealMLP 48 epochs x 3 seeds was the 09-03 blend partner

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
kind, seed = 'tabm', 42
oof, tpred = np.zeros(len(y)), np.zeros(len(Xte))
for f, (itr, iva) in enumerate(StratifiedKFold(NFOLDS, shuffle=True, random_state=SEED).split(X, y)):
    m = NN(kind, EPOCHS[kind], seed).fit(Xd.iloc[itr], y.iloc[itr])        # ~25 min per fold on a laptop GPU
    oof[iva] = m.predict_proba(Xd.iloc[iva])[:, 1]; tpred += m.predict_proba(Xted)[:, 1] / NFOLDS
    print('fold%d auc=%.5f' % (f, roc_auc_score(y.iloc[iva], oof[iva])))
print('oof=%.5f' % roc_auc_score(y, oof))
np.save(OOF / f'oof_{kind}_s{seed}.npy', oof); np.save(TEST / f'test_{kind}_s{seed}.npy', tpred)

## The yekenot RealMLP port (`src/yeke_realmlp_stream.py`)
Vladimir Demidov's public PyTorch RealMLP (LB 0.94595) treats numerics as categorical embeddings, adds pair target encodings
and original-data means, and trains with PBLD. We kept his code verbatim and moved only the best-epoch selection to an inner
split. It is the one NN that kept a positive weight (0.035): different inputs, not a different architecture, is what made it
useful.

## What we learned
- A better NN is not a better blend partner. RealMLP input tweaks (Age categorical + commute TE) gave +0.00045 alone and
  +0.00000 in the stack: it had become a copy of the tree.
- 5 TabM seeds bagged: 0.9440 -> 0.94417. Seeds buy variance, not information.
- AutoGluon best_quality (7.8 h on Kaggle) scored 0.94194 on our folds: default binning never sees exact values. Excluded.